In [ ]:
import hashlib
from pathlib import Path

import pandas as pd

In [ ]:
def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            candidate = candidate.parent
        if candidate.exists() and (candidate / "paper_submissions_manifest.csv").exists():
            return candidate
        if candidate.exists() and candidate.is_dir():
            for path in candidate.glob("*/paper_submissions_manifest.csv"):
                return path.parent
    raise FileNotFoundError("paper_submissions_manifest.csv")

repo_root = find_repo_root()
manifest = pd.read_csv(repo_root / "paper_submissions_manifest.csv", dtype=str).fillna("")
output_dir = Path("/kaggle/working/paper_submissions") if Path("/kaggle/working").exists() and str(repo_root).replace("\\", "/").startswith("/kaggle/") else repo_root / "paper_submissions"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

records = []
selected_path = None
for row in manifest.to_dict("records"):
    record = dict(row)
    if row["source_status"] == "archived_csv":
        source = repo_root / row["input_csv"]
        destination = output_dir / f"{row['slug']}.csv"
        destination.write_bytes(source.read_bytes())
        submission = pd.read_csv(destination)[["image_id", "cluster"]].copy()
        actual = sha256_file(destination)
        record["reproduced_csv"] = str(destination)
        record["reproduced_sha256"] = actual
        record["reproduced_rows"] = len(submission)
        record["reproduction_status"] = "ok" if actual == row["sha256"] else "hash_mismatch"
        if row["slug"] == "salamander_eva02_aliked_local_link_refinement":
            selected_path = destination
    else:
        notebook_path = repo_root / row["notebook"] if row["notebook"] else Path("")
        record["reproduced_csv"] = ""
        record["reproduced_sha256"] = ""
        record["reproduced_rows"] = ""
        record["reproduction_status"] = "run_original_notebook" if notebook_path.exists() else "missing_notebook"
    records.append(record)

report = pd.DataFrame(records)
report.to_csv(output_dir / "reproduction_report.csv", index=False, lineterminator="\n")
if selected_path is not None:
    final_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() and str(repo_root).replace("\\", "/").startswith("/kaggle/") else repo_root / "submission.csv"
    final_path.write_bytes(selected_path.read_bytes())
else:
    final_path = ""
print(len(report))
print(report["reproduction_status"].value_counts().to_string())
print(final_path)
if final_path:
    print(sha256_file(final_path))